In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

os.makedirs(project_path, exist_ok=True)

print("Project folder created successfully!")
print(project_path)

Project folder created successfully!
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine


In [3]:
folders = [
    "dataset",
    "models",
    "images",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("All folders created successfully!")

All folders created successfully!


In [4]:
import os

for root, dirs, files in os.walk(project_path):
    print(root)
    for d in dirs:
        print("   📁", d)

/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine
   📁 dataset
   📁 models
   📁 images
   📁 notebooks
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/images
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/notebooks


In [5]:
!pip install tensorflow streamlit scikit-learn nltk pandas numpy google-generativeai matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.3 MB/s eta 0:00:00


In [6]:
import tensorflow as tf
import pandas as pd
import numpy as np
import nltk
import streamlit
import sklearn

print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
!wget https://raw.githubusercontent.com/dair-ai/emotion_dataset/master/data/training.csv -O /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset/dataset.csv

--2026-07-04 08:43:22--  https://raw.githubusercontent.com/dair-ai/emotion_dataset/master/data/training.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-04 08:43:22 ERROR 404: Not Found.



In [8]:
import pandas as pd

data = {
    "text": [
        "I am very happy today",
        "I feel sad and lonely",
        "I am extremely angry",
        "I am scared of exams",
        "I love learning AI",
        "I feel disappointed",
        "I am excited about my project",
        "I feel nervous before presentation",
        "I am frustrated with coding",
        "I feel confident today",
        "I am depressed",
        "I feel joyful",
        "I am worried about results",
        "I am motivated to study",
        "I feel relaxed",
        "I am stressed",
        "Everything is wonderful",
        "I hate this situation",
        "I am feeling amazing",
        "I don't know what to do"
    ],
    "emotion": [
        "happy",
        "sad",
        "anger",
        "fear",
        "happy",
        "sad",
        "happy",
        "fear",
        "anger",
        "happy",
        "sad",
        "happy",
        "fear",
        "happy",
        "happy",
        "fear",
        "happy",
        "anger",
        "happy",
        "sad"
    ]
}

df = pd.DataFrame(data)
df.head()

,text,emotion
0,I am very happy today,happy
1,I feel sad and lonely,sad
2,I am extremely angry,anger
3,I am scared of exams,fear
4,I love learning AI,happy


In [9]:
dataset_path = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset/dataset.csv"

df.to_csv(dataset_path, index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [10]:
df = pd.read_csv(dataset_path)

print(df.head())

                    text emotion
0  I am very happy today   happy
1  I feel sad and lonely     sad
2   I am extremely angry   anger
3   I am scared of exams    fear
4     I love learning AI   happy


In [11]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Read dataset
df = pd.read_csv(dataset_path)

# Text and labels
texts = df['text']
labels = df['emotion']

# Tokenizer
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
X = pad_sequences(sequences, maxlen=20)

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(labels)

print("Text preprocessing completed successfully!")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Text preprocessing completed successfully!
Shape of X: (20, 20)
Shape of y: (20,)


In [12]:
import pickle

# Save tokenizer
with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Tokenizer and Label Encoder saved successfully!")

Tokenizer and Label Encoder saved successfully!


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=5000, output_dim=64),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(len(encoder.classes_), activation='softmax')
])

# Build the model by specifying the input shape
model.build(input_shape=(None, 20))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 20, 64)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,308 (1.49 MB)

 Trainable params: 390,308 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X,
    y,
    epochs=30,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step - accuracy: 0.4375 - loss: 1.3789 - val_accuracy: 0.5000 - val_loss: 1.3720
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4375 - loss: 1.3582 - val_accuracy: 0.5000 - val_loss: 1.3634
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4375 - loss: 1.3319 - val_accuracy: 0.5000 - val_loss: 1.3422
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4375 - loss: 1.3384 - val_accuracy: 0.5000 - val_loss: 1.3237
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.4375 - loss: 1.3034 - val_accuracy: 0.5000 - val_loss: 1.2999
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4375 - loss: 1.2919 - val_accuracy: 0.5000 - val_loss: 1.2874
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4375 - loss: 1.3007 - val_accuracy: 0.5000 - val_loss: 1.2874
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4375 - loss: 1.2379 - val_accuracy: 0.5000 - val_loss: 1.2939

In [16]:
model.save("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models/bilstm_model.h5")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [17]:
import os

model_path = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models"

print("Files in models folder:")
for file in os.listdir(model_path):
    print(file)

Files in models folder:
tokenizer.pkl
label_encoder.pkl
bilstm_model.h5


In [18]:
app_code = '''
import streamlit as st
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load Model
model = load_model("models/bilstm_model.h5")

# Load Tokenizer
with open("models/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load Label Encoder
with open("models/label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

st.set_page_config(page_title="Emotion Detection", page_icon="😊")

st.title("😊 Emotion Detection Learning Support Engine")

st.write("Enter any sentence to predict the emotion.")

user_input = st.text_area("Enter Text")

if st.button("Predict Emotion"):

    if user_input.strip() != "":

        sequence = tokenizer.texts_to_sequences([user_input])

        padded = pad_sequences(sequence,maxlen=20)

        prediction = model.predict(padded)

        emotion = encoder.inverse_transform([np.argmax(prediction)])[0]

        confidence = np.max(prediction)*100

        st.success(f"Predicted Emotion : {emotion}")

        st.info(f"Confidence : {confidence:.2f}%")

    else:

        st.warning("Please enter some text.")
'''

with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/app.py","w") as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [19]:
import os

project = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

print(os.listdir(project))

['dataset', 'models', 'images', 'notebooks', 'app.py']


In [20]:
requirements = """streamlit
tensorflow
numpy
pandas
scikit-learn
nltk
google-generativeai
"""

with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created successfully!")

✅ requirements.txt created successfully!


In [21]:
gitignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
.env
"""

with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/.gitignore", "w") as f:
    f.write(gitignore)

print("✅ .gitignore created successfully!")

✅ .gitignore created successfully!


In [22]:
readme = """
# Emotion Detection Learning Support Engine

## Project Description
This project predicts human emotions from text using a Bidirectional LSTM (BiLSTM) deep learning model.

## Features
- Emotion prediction from text
- BiLSTM neural network
- Streamlit web interface
- Confidence score display

## Technologies Used
- Python
- TensorFlow
- Streamlit
- Pandas
- NumPy
- Scikit-learn
"""

with open("/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/README.md", "w") as f:
    f.write(readme)

print("✅ README.md created successfully!")

✅ README.md created successfully!


In [23]:
import os

project = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

print("Project Files:\n")

for file in os.listdir(project):
    print(file)

Project Files:

dataset
models
images
notebooks
app.py
requirements.txt
.gitignore
README.md


In [24]:
import os

project = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

for root, dirs, files in os.walk(project):
    print(root)
    for file in files:
        print("   📄", file)

/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine
   📄 app.py
   📄 requirements.txt
   📄 .gitignore
   📄 README.md
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset
   📄 dataset.csv
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models
   📄 tokenizer.pkl
   📄 label_encoder.pkl
   📄 bilstm_model.h5
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/images
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/notebooks


In [25]:
import os

project = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

print("Project exists:", os.path.exists(project))

for root, dirs, files in os.walk(project):
    print(root)
    print("Folders:", dirs)
    print("Files:", files)
    print("-"*40)

Project exists: True
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine
Folders: ['dataset', 'models', 'images', 'notebooks']
Files: ['app.py', 'requirements.txt', '.gitignore', 'README.md']
----------------------------------------
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset
Folders: []
Files: ['dataset.csv']
----------------------------------------
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models
Folders: []
Files: ['tokenizer.pkl', 'label_encoder.pkl', 'bilstm_model.h5']
----------------------------------------
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/images
Folders: []
Files: []
----------------------------------------
/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/notebooks
Folders: []
Files: []
----------------------------------------


In [26]:
import os

project = "/content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine"

for root, dirs, files in os.walk(project):
    print(f"\n📂 {root}")
    for file in files:
        print("   📄", file)


📂 /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine
   📄 app.py
   📄 requirements.txt
   📄 .gitignore
   📄 README.md

📂 /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/dataset
   📄 dataset.csv

📂 /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/models
   📄 tokenizer.pkl
   📄 label_encoder.pkl
   📄 bilstm_model.h5

📂 /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/images

📂 /content/drive/MyDrive/Emotion-Detection-Learning-Support-Engine/notebooks
